- Importando bibliotecas para manipulação de arquivos, dados tabulares, processamento de imagens e barra de progresso

In [ ]:
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm


- Definir o diretório principal onde o dataset EMNIST está armazenado no Kaggle 
- Construir os caminhos completos e seguros para os arquivos de treino, teste e mapeamento

In [ ]:
DATA_DIR = "/kaggle/input/datasets/crawford/emnist"

TRAIN_CSV = os.path.join(DATA_DIR, "emnist-byclass-train.csv")
TEST_CSV = os.path.join(DATA_DIR, "emnist-byclass-test.csv")
MAPPING = os.path.join(DATA_DIR, "emnist-byclass-mapping.txt")


- Define e cria o diretório de saída onde os dados processados serão salvos
- Cria um dicionário vazio para armazenar o mapeamento original (ID da classe -> Caractere) 
- Abre o arquivo de texto que contém o mapeamento de classes 
- Remove espaços extras e separando o ID da classe e o código ASCII 
- Converte as strings lidas do texto para números inteiros 
- Converte o código ASCII de volta para o caractere real e salvando no dicionário 
- Prepara variáveis para filtrar classes específicas e criar uma nova numeração sequencial

In [ ]:

OUTPUT = "/kaggle/working/dataset"
os.makedirs(OUTPUT, exist_ok=True)

mapping = {}

with open(MAPPING) as f:
    for line in f:
        idx, ascii_code = line.strip().split()
        idx = int(idx)
        ascii_code = int(ascii_code)

        mapping[idx] = chr(ascii_code)

valid_classes = {}

new_label = 0

- Mapeamento original (ID antigo e o caractere) 
- Verifica se o caractere atual é um dígito (0 a 9) 
- Salva a classe válida com seu novo rótulo (label) sequencial 
- Incrementa o contador para o próximo rótulo 
- Verifica se o caractere é uma letra maiúscula (A a Z) 
- Salva a classe válida com seu novo rótulo sequencial 
- Incrementa o contador para o próximo rótulo

In [ ]:

for old, char in mapping.items():

    if char.isdigit():

        valid_classes[old] = {
            "char": char,
            "label": new_label
        }

        new_label += 1

    elif char.isupper():

        valid_classes[old] = {
            "char": char,
            "label": new_label
        }

        new_label += 1

- Imprime a quantidade de classe encontrada

In [ ]:
print("Classes encontradas:", len(valid_classes))

- Lê o arquivo CSV (Comma-Separated Values) completo para a memória. O 'header=None' avisa que a primeira linha já é de dados.
- Itera sobre todas as linhas do CSV de forma rápida, mostrando uma barra de progresso (tqdm) 
- No EMNIST, a primeira coluna (índice 0) contém o número da classe (rótulo) 
- Se for uma das classes que decidimos ignorar (como letras minúsculas), pula para a próxima iteração 
- Resgata qual é o caractere real (ex: 'A', '3') mapeado para este rótulo 
- Constrói o caminho para salvar a imagem separada por pastas de categorias 
- Garante que a pasta desse caractere exista 
- Extrai os 784 valores de pixels (da segunda coluna até o final da linha) 
- Cria uma tela em branco no formato "L" (escala de cinza) de 28x28 e preenche com os pixels
- Essas duas linhas corrigem a orientação para que o caractere fique em pé 
- Se a imagem tiver fundo branco (média dos pixels > 127), invertemos as cores. 
- Define o nome do arquivo usando um contador interno da função para não haver nomes duplicados 
- Salva a imagem em disco
- Incrementa o contador em 1 para a próxima imagem 

In [ ]:

def process(csv_file, split):

    df = pd.read_csv(csv_file, header=None)

    for row in tqdm(df.itertuples(index=False), total=len(df)):

        old_label = row[0]

        if old_label not in valid_classes:
            continue

        char = valid_classes[old_label]["char"]

        folder = os.path.join(
            OUTPUT,
            split,
            char
        )

        os.makedirs(folder, exist_ok=True)

        pixels = list(row[1:])

        img = Image.new("L", (28,28))
        img.putdata(pixels)

        img = img.transpose(Image.TRANSPOSE)
        img = img.transpose(Image.FLIP_LEFT_RIGHT)

        arr = list(img.getdata())
        if sum(arr) / len(arr) > 127:
            arr = [255 - p for p in arr]
            img.putdata(arr)

        filename = os.path.join(
            folder,
            f"{tqdm.format_num(process.counter)}.png"
        )

        img.save(filename)

        process.counter += 1

- Imprime conversão de treino/teste 

In [ ]:
print("Convertendo treino...")
process(TRAIN_CSV, "train")

print("Convertendo teste...")
process(TEST_CSV, "test")